In [94]:
import sympy as sp

In [95]:
from functools import lru_cache

@lru_cache(maxsize = None)
def get_ckn(k: int, n: int, p):
    if k<0 or k>n:
        return sp.Integer(0)
    if n == 0:
        return sp.Integer(1) if k==0 else sp.Integer(0)
    return get_ckn(k-1, n-1, p)/(2*p) + (k+1)*get_ckn(k+1, n-1, p)


### Implementation of overlap integrals

In [96]:
alpha, beta = sp.symbols('alpha beta', real = True, positive = True)
Ax, Ay, Az = sp.symbols('Ax Ay Az', real = True)
Bx, By, Bz = sp.symbols('Bx By Bz', real = True)

p = alpha + beta
q = alpha*beta/p
RAB2 = (Ax-Bx)**2 + (Ay-By)**2 + (Az-Bz)**2


In [97]:
S00 = (sp.sqrt(sp.pi/p))**3 * sp.exp(-q*RAB2)
S00

pi**(3/2)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(3/2)

In [98]:
def build_derivative_table(base_integral, Lmax, Avars, Bvars, indices):
    Ax, Ay, Az = Avars
    Bx, By, Bz = Bvars
    
    @lru_cache(maxsize=None)
    def derivative(i,j,k,l,m,n):
        if (i,j,k,l,m,n) == (0,0,0,0,0,0):
            return base_integral
        if i>0:
            return sp.diff(derivative(i-1,j,k,l,m,n),Ax)
        if j>0:
            return sp.diff(derivative(i,j-1,k,l,m,n),Ay)
        if k>0:
            return sp.diff(derivative(i,j,k-1,l,m,n),Az) 
        if l>0:
            return sp.diff(derivative(i,j,k,l-1,m,n),Bx)
        if m>0:
            return sp.diff(derivative(i,j,k,l,m-1,n),By)
        return sp.diff(derivative(i,j,k,l,m,n-1),Bz) 
    
    derivatives_dict = {}
    for (i,j,k,l,m,n) in indices:
        derivatives_dict[(i,j,k,l,m,n)] = derivative(i,j,k,l,m,n).simplify()
    return derivatives_dict

## Generate all induces up to Lmax

In [99]:
Lmax = 1
def l_to_ijk(L):
    IJK = []
    for I in range(L, -1, -1):
        for J in range(L - I, -1, -1):
            IJK.append((I, J, L - I - J))
    return sorted(IJK, reverse=True)

integral_indices=[]
ijk = [t for L in range(Lmax+1) for t in l_to_ijk(L)]
for i in ijk:
    for j in ijk:
        integral_indices.append(i+j)
integral_indices

[(0, 0, 0, 0, 0, 0),
 (0, 0, 0, 1, 0, 0),
 (0, 0, 0, 0, 1, 0),
 (0, 0, 0, 0, 0, 1),
 (1, 0, 0, 0, 0, 0),
 (1, 0, 0, 1, 0, 0),
 (1, 0, 0, 0, 1, 0),
 (1, 0, 0, 0, 0, 1),
 (0, 1, 0, 0, 0, 0),
 (0, 1, 0, 1, 0, 0),
 (0, 1, 0, 0, 1, 0),
 (0, 1, 0, 0, 0, 1),
 (0, 0, 1, 0, 0, 0),
 (0, 0, 1, 1, 0, 0),
 (0, 0, 1, 0, 1, 0),
 (0, 0, 1, 0, 0, 1)]

In [100]:
derivatives_dict = build_derivative_table(S00, Lmax, (Ax,Ay,Az), (Bx,By,Bz), integral_indices)

## Generate only canonical indices

In [101]:
from itertools import permutations

S3 = list(permutations((0,1,2)))

def permute(ijk, permutation):
    return tuple(ijk[i] for i in permutation)

def l_to_ijk(L):
    return [(i, j, L - i - j) for i in range(L + 1) for j in range(L+1-i)]

def get_canonical_indices(La, Lb):
    canonical_indices = set()
    for (i,j,k) in l_to_ijk(La):
        for (l,m,n) in l_to_ijk(Lb):
            orbit_ijklmn = set()
            for permutation in S3:
                permuted_ijk = permute((i,j,k), permutation)
                permuted_lmn = permute((l,m,n), permutation)
                idx = permuted_ijk + permuted_lmn
                orbit_ijklmn.add(idx)
                idx = permuted_lmn + permuted_ijk
                orbit_ijklmn.add(idx)
            canonical_indices.add(min(orbit_ijklmn))
    return canonical_indices
                
indices = set()
for La in range(Lmax+1):
    for Lb in range(Lmax+1):
        indices.update(get_canonical_indices(La, Lb))   
indices = sorted(indices)
canonical_to_rep = {idx: num for num, idx in enumerate(indices)}
canonical_to_rep

{(0, 0, 0, 0, 0, 0): 0,
 (0, 0, 0, 0, 0, 1): 1,
 (0, 0, 1, 0, 0, 1): 2,
 (0, 0, 1, 0, 1, 0): 3}

### Generate permutation mappings
For each set of indices generate a permutation that maps the given set to a canonical index.
Construct the dictionary with key = ijklmn and values = (permutation, swap)

In [102]:
ijklmn_to_rep = {}

for La in range(Lmax+1):
    for Lb in range(Lmax+1):
        ijk_list = l_to_ijk(La)
        lmn_list = l_to_ijk(Lb)
        for ijk in ijk_list:
            for lmn in lmn_list:
                ijklmn = ijk + lmn
                for canonical in indices:
                    for perm in S3:
                        pijk = permute(ijk, perm)
                        plmn = permute(lmn, perm)
                        pijklmn = pijk + plmn
                        if pijklmn == canonical:
                            ijklmn_to_rep[ijklmn] = (canonical_to_rep[canonical],perm, False)
                        plmnijk = plmn + pijk
                        if plmnijk == canonical:
                            ijklmn_to_rep[ijklmn] = (canonical_to_rep[canonical],perm, True)



In [103]:
ijklmn_to_rep

{(0, 0, 0, 0, 0, 0): (0, (2, 1, 0), True),
 (0, 0, 0, 0, 0, 1): (1, (1, 0, 2), False),
 (0, 0, 0, 0, 1, 0): (1, (2, 0, 1), False),
 (0, 0, 0, 1, 0, 0): (1, (2, 1, 0), False),
 (0, 0, 1, 0, 0, 0): (1, (1, 0, 2), True),
 (0, 1, 0, 0, 0, 0): (1, (2, 0, 1), True),
 (1, 0, 0, 0, 0, 0): (1, (2, 1, 0), True),
 (0, 0, 1, 0, 0, 1): (2, (1, 0, 2), True),
 (0, 0, 1, 0, 1, 0): (3, (0, 2, 1), True),
 (0, 0, 1, 1, 0, 0): (3, (1, 2, 0), True),
 (0, 1, 0, 0, 0, 1): (3, (0, 2, 1), False),
 (0, 1, 0, 0, 1, 0): (2, (2, 0, 1), True),
 (0, 1, 0, 1, 0, 0): (3, (2, 1, 0), True),
 (1, 0, 0, 0, 0, 1): (3, (1, 2, 0), False),
 (1, 0, 0, 0, 1, 0): (3, (2, 1, 0), False),
 (1, 0, 0, 1, 0, 0): (2, (2, 1, 0), True)}

In [104]:
def get_integral_expressions(integral_indices, derivatives_dict, alpha, beta):
    integral_expressions = {}

    for (i,j,k,l,m,n) in integral_indices:

        ci = [get_ckn(o, i, alpha) for o in range(i + 1)]
        cj = [get_ckn(p, j, alpha) for p in range(j + 1)]
        ck = [get_ckn(q, k, alpha) for q in range(k + 1)]

        cl = [get_ckn(r, l, beta) for r in range(l + 1)]
        cm = [get_ckn(s, m, beta) for s in range(m + 1)]
        cn = [get_ckn(t, n, beta) for t in range(n + 1)]

        expr = 0
        for o, co in enumerate(ci):
            for p, cp in enumerate(cj):
                for q, cq in enumerate(ck):
                    for r, cr in enumerate(cl):
                        for s, cs in enumerate(cm):
                            for t, ct in enumerate(cn):
                                expr += (
                                    co * cp * cq * cr * cs * ct
                                    * derivatives_dict[(o, p, q, r, s, t)]
                                )

        integral_expressions[(i,j,k,l,m,n)] = sp.simplify(expr)
    return integral_expressions

In [105]:
integral_expressions = get_integral_expressions(indices, derivatives_dict, alpha, beta)

In [106]:
Dx, Dy, Dz, D2, P, Q = sp.symbols(
    'Dx Dy Dz D2 P Q',
    real=True,
)

subsdict = {
    Ax - Bx: Dx,
    Ay - By: Dy,
    Az - Bz: Dz,
    (Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2: RAB2,
    alpha + beta: P,
    alpha * beta: Q,
}
for key, value in integral_expressions.items():
    integral_expressions[key] = value.subs(subsdict)
    print(key, integral_expressions[key])

(0, 0, 0, 0, 0, 0) pi**(3/2)*exp(-Q*(Dx**2 + Dy**2 + Dz**2)/P)/P**(3/2)
(0, 0, 0, 0, 0, 1) pi**(3/2)*Dz*alpha*exp(-Q*(Dx**2 + Dy**2 + Dz**2)/P)/P**(5/2)
(0, 0, 1, 0, 0, 1) pi**(3/2)*(-2*Dz**2*Q + P)*exp(-Q*(Dx**2 + Dy**2 + Dz**2)/P)/(2*P**(7/2))
(0, 0, 1, 0, 1, 0) -pi**(3/2)*Dy*Dz*Q*exp(-Q*(Dx**2 + Dy**2 + Dz**2)/P)/P**(7/2)


In [107]:
from sympy.printing.numpy import NumPyPrinter, _known_constants_numpy, _known_functions_numpy

printer = NumPyPrinter()
printer._module = "np"
printer.known_functions = { k:f"np.{v}" for k,v in _known_functions_numpy.items()}
printer.known_constants = {k: f"np.{v}" for k, v in _known_constants_numpy.items()}


In [108]:
def write_onel_module(path, name = "S", variables = None,  integral_expressions = None):
    lines = ["import numpy as np\n",
             "from numba import njit\n",
             "@njit(cache=True, fastmath=True)\n",
              f"def {name}({variables}):\n"]
    
    for rep, value in enumerate(integral_expressions.values()):
        lines.append(f"    if rep == {rep}:\n")
        lines.append(f"        return {printer.doprint(value)}\n")
    lines.append("    raise KeyError((i,j,k,l,m,n))")
    path = path / (name + ".py")
    with open(path, "w", encoding="utf-8") as f:
        f.write("".join(lines))

In [109]:
from pathlib import Path
path = Path.cwd() / "src_live"
write_onel_module(path, name = "S", variables = "i, j, k, l, m, n, Dx, Dy, Dz, P, Q, RAB2", integral_expressions = integral_expressions)

In [111]:
import timeit
%timeit S.S(0, 0, 1, 0, 1, 0, Dx, Dy, Dz, P, Q, RAB2)


551 ns ± 5.31 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


## Overlap shell pair

In [ ]:
IJKLMN_TO_REP = {(0, 0, 0, 0, 0, 0): (0, (2, 1, 0), True),
 (0, 0, 0, 0, 0, 1): (1, (1, 0, 2), False),
 (0, 0, 0, 0, 1, 0): (1, (2, 0, 1), False),
 (0, 0, 0, 1, 0, 0): (1, (2, 1, 0), False),
 (0, 0, 1, 0, 0, 0): (1, (1, 0, 2), True),
 (0, 1, 0, 0, 0, 0): (1, (2, 0, 1), True),
 (1, 0, 0, 0, 0, 0): (1, (2, 1, 0), True),
 (0, 0, 1, 0, 0, 1): (2, (1, 0, 2), True),
 (0, 0, 1, 0, 1, 0): (3, (0, 2, 1), True),
 (0, 0, 1, 1, 0, 0): (3, (1, 2, 0), True),
 (0, 1, 0, 0, 0, 1): (3, (0, 2, 1), False),
 (0, 1, 0, 0, 1, 0): (2, (2, 0, 1), True),
 (0, 1, 0, 1, 0, 0): (3, (2, 1, 0), True),
 (1, 0, 0, 0, 0, 1): (3, (1, 2, 0), False),
 (1, 0, 0, 0, 1, 0): (3, (2, 1, 0), False),
 (1, 0, 0, 1, 0, 0): (2, (2, 1, 0), True)}

def overlap_shell_pair(exp_a, coeff_a, norm_a, center_a,
                       exp_b, coeff_b, norm_b, center_b,
                       ijklmn):
    dim_a = norm_a.shape[1]
    dim_b = norm_b.shape[1]
    out = np.zeros((dim_a, dim_b), dtype = np.float64)
    D2 = (center_a[0]-center_b[0])**2 + (center_a[1]-center_b[1])**2 + (center_a[2]-center_b[2])**2

    for idx in range(ijklmn.shape[0]):
        ia = idx // dim_b
        ib = idx % dim_b
        ijklkmn_tuple = ijklmn[idx]
        rep, perm, swap = IJKLMN_TO_REP[ijklkmn_tuple]
        if swap == False:
            A,B = center_a, center_b
        else:
            A,B = center_b, center_a
        Dx = A[perm[0]] - B[perm[0]]
        Dy = A[perm[1]] - B[perm[1]]
        Dz = A[perm[2]] - B[perm[2]]
        val = 0.0
        for pa in range(exp_a.shape[0]):
            alpha_a = exp_a[pa]
            ca = coeff_a[pa] * norm_a[pa, ia]
            for pb in range(exp_b.shape[0]):
                alpha_b = exp_b[pb]
                cb = coeff_b[pb] * norm_b[pb, ib]
                P = alpha_a + alpha_b
                Q = alpha_a * alpha_b
                val += ca * cb * S(i,j,k,l,m,n,Dx,Dy,Dz,D2,P,Q)

        out[ia, ib] = val
    return out
    